# Fix Git HTTPS certificate errors (Terminal-only commands)

This notebook fixes:

```

fatal: unable to access 'https://<your-gitlab-host>/<namespace>/<repo>.git/': server certificate verification failed. CAfile: none CRLfile: none
```

**Scope:** HTTPS only, using **terminal commands** (bash). No SSH.



## Quick diagnostics

```bash
# Show the CA bundle Git uses globally (may be empty)
git config --show-origin --get http.sslcainfo || true

# Show host-specific CA bundle (example host below)
git config --show-origin --get 'http.https://git.dsa.missouri.edu/.sslCAInfo' || true

# Show Python certifi path (useful as a base bundle)
python -c "import certifi; print(certifi.where())" || true
```

## Step 1 — Point Git to Python's `certifi` bundle (simple case)

```bash
# Point Git (global) to the certifi CA file (works for public CAs)
git config --global http.sslcainfo "$(python -c 'import certifi; print(certifi.where())')"

# Optional: test clone (replace URL with your repo)
# git clone https://git.dsa.missouri.edu/test/f25/f25dsa8420_sknnh.git || true

```


> If verification still fails, your Git server likely uses an **internal CA**. Proceed to Step 2.

## Step 2 — Build a host-specific combined CA bundle (recommended)

```bash
# --- Configure your Git host and target bundle path ---
GIT_HOST="git.dsa.missouri.edu"
CABUNDLE="$HOME/.git-cabundle.pem"

# 0) (Optional) clear previous global override
git config --global --unset http.sslcainfo 2>/dev/null || true

# 1) Start the bundle from certifi (copy, do not move)
cp "$(python -c 'import certifi; print(certifi.where())')" "$CABUNDLE"

# 2) Append the presented certificate chain from the Git host
openssl s_client -showcerts -servername "$GIT_HOST" -connect "$GIT_HOST:443" </dev/null 2>/dev/null   | sed -n '/BEGIN CERTIFICATE/,/END CERTIFICATE/p' >> "$CABUNDLE"

# 3) Tell Git to use this bundle ONLY for that host
git config --global "http.https://$GIT_HOST/.sslCAInfo" "$CABUNDLE"

echo "Host-scoped CA bundle set for $GIT_HOST -> $CABUNDLE"
```

### Verify the fix

```bash
# Use verbose CURL output to confirm TLS handshake and certificate validation.
GIT_HOST="git.dsa.missouri.edu"
TEST_URL="https://$GIT_HOST/dsa-8420/f25/f25dsa8420_{MU_SSO_ID}.git"

echo "Probing remote:" "$TEST_URL"
GIT_CURL_VERBOSE=1 git ls-remote "$TEST_URL" || true
```

### Alternative (if `openssl` unavailable)


Export your Git server's **root/intermediate** certificates from a browser as PEM files and append them:

```bash
GIT_HOST="git.dsa.missouri.edu"
CABUNDLE="$HOME/.git-cabundle.pem"

# Start fresh from certifi
cp "$(python -c 'import certifi; print(certifi.where())')" "$CABUNDLE"

# Append one or more PEM files you exported (adjust paths)
cat ~/Downloads/gitlab_root_or_intermediate.pem >> "$CABUNDLE"

# Configure host-scoped CA bundle
git config --global "http.https://$GIT_HOST/.sslCAInfo" "$CABUNDLE"
```
Then re-run the **Verify** step.


## Safety notes


- **Do not disable SSL verification** globally (e.g., `git config --global http.sslVerify false` or `GIT_SSL_NO_VERIFY=1`). Use only as a last-resort temporary bypass.

- Keeping the fix **host-scoped** ensures other HTTPS connections remain unaffected.

- If your org rotates certificates/CA, rebuild `~/.git-cabundle.pem` by re-running Step 2.


## Troubleshooting checklist


- Confirm the host you configured matches your remote URL (`git.dsa.missouri.edu`).

- Inspect which CA bundle Git uses for that host:

  ```bash
  git config --show-origin --get 'http.https://git.dsa.missouri.edu/.sslCAInfo'
  ```

- Use verbose probe to inspect TLS details:

  ```bash
  GIT_CURL_VERBOSE=1 git ls-remote https://git.dsa.missouri.edu/namespace/repo.git
  ```

- If failures persist, ask your admin for the correct **root/intermediate** CA PEMs and append them to `~/.git-cabundle.pem`.


## Summary


1. Try setting `http.sslcainfo` to the **certifi** bundle.

2. For enterprise Git servers, build a **host-specific combined bundle** and configure:

   ```bash
   git config --global 'http.https://<your-host>/.sslCAInfo' ~/.git-cabundle.pem
   ```

This preserves HTTPS security without disabling verification.
